# ProveTok Stage0-4 Output Analysis (Colab)

???????**450/450 ?????**?

???????
- **single**??????????`summary.csv / run_meta.json / cases/`?
- **sweep**?????????????????????

**??????????**
- `cp_strict = true`
- `r2_mode = ratio`
- `r2_min_support_ratio = 0.8`
- `tau_iou = 0.05`
- `anatomy_spatial_routing = true`
- `r2_skip_bilateral = true`
- `r4_disabled = true`
- `r5_fallback_disabled = true`


In [ ]:
!pip install -q pandas numpy matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Config (change before running)
MODE = 'single'   # 'single' | 'sweep'

# Single-run directory for the 450/450 main experiment
OUT_DIR = '/content/drive/MyDrive/Data/outputs_stage0_4_450_128'
EXPECTED_CASES_MAP = 'ctrate=450,radgenome=450'

# Optional sweep root
SWEEP_ROOT = '/content/drive/MyDrive/Data/outputs_stage0_4_r2_sweeps'
SWEEP_GLOB = '*'

from pathlib import Path
import json
from collections import Counter
import random
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

OUT_PATH = Path(OUT_DIR)
SWEEP_PATH = Path(SWEEP_ROOT)

if MODE == 'single':
    assert OUT_PATH.exists(), f'OUT_DIR not found: {OUT_PATH}'
    print('OUT_DIR:', OUT_PATH)
    print('EXPECTED_CASES_MAP:', EXPECTED_CASES_MAP)
elif MODE == 'sweep':
    assert SWEEP_PATH.exists(), f'SWEEP_ROOT not found: {SWEEP_PATH}'
    print('SWEEP_ROOT:', SWEEP_PATH)
    _runs = sorted([p for p in SWEEP_PATH.glob(SWEEP_GLOB) if p.is_dir()])
    print(f'  SWEEP_GLOB="{SWEEP_GLOB}"  Found {len(_runs)} runs')
    for r in _runs:
        print(f'    {r.name}')
    OUT_PATH = _runs[0] if _runs else OUT_PATH
    print(f'
  Per-case analysis: {OUT_PATH.name}  (modify OUT_PATH below if needed)')


## 1. ?????450/450?

??????? `validate_stage0_4_outputs.py`????? `ctrate=450`?`radgenome=450`?

In [ ]:
%cd /content
!test -d ProveTok_ACM || git clone https://github.com/Ricardo-ChenTY/ProveTok_ACM.git -q
%cd /content/ProveTok_ACM

In [ ]:
result = subprocess.run(
    [
        'python', 'validate_stage0_4_outputs.py',
        '--out_dir', str(OUT_PATH),
        '--datasets', 'ctrate,radgenome',
        '--expected_cases_map', EXPECTED_CASES_MAP,
        '--save_report', str(OUT_PATH / 'validation_report.json'),
    ],
    capture_output=True, text=True, cwd='/content/ProveTok_ACM'
)
out = result.stdout + result.stderr
print(out[-4000:] if len(out) > 4000 else out)


In [ ]:
rmp = OUT_PATH / 'run_meta.json'
vrp = OUT_PATH / 'validation_report.json'

if rmp.exists():
    run_meta = json.loads(rmp.read_text(encoding='utf-8'))
    print('=== run_meta.json ===')
    for k, v in run_meta.items():
        if not isinstance(v, dict):
            print(f'  {k}: {v}')
        else:
            print(f'  [{k}]')
            for kk, vv in v.items():
                print(f'    {kk}: {vv}')
else:
    run_meta = {}
    print('run_meta.json not found')

validation_rows = []
if vrp.exists():
    validation_rows = json.loads(vrp.read_text(encoding='utf-8'))
    failed = sum(1 for row in validation_rows if not row.get('passed', False))
    print('
=== validation_report.json ===')
    print(f'  validated rows: {len(validation_rows)}')
    print(f'  failed rows:    {failed}')
else:
    print('validation_report.json not found')

checks = []
checks.append(('cp_strict == true', run_meta.get('cp_strict') is True))
checks.append(('r2_mode == ratio', run_meta.get('r2_mode') == 'ratio'))
checks.append(('r2_min_support_ratio == 0.8', float(run_meta.get('r2_min_support_ratio', -1)) == 0.8))
checks.append(('tau_iou == 0.05', float(run_meta.get('tau_iou', -1)) == 0.05))
checks.append(('r2_skip_bilateral == true', run_meta.get('r2_skip_bilateral') is True))
checks.append(('anatomy_spatial_routing == true', run_meta.get('anatomy_spatial_routing') is True))
checks.append(('r4_disabled == true', run_meta.get('r4_disabled') is True))
checks.append(('r5_fallback_disabled == true', run_meta.get('r5_fallback_disabled') is True))

for ds in ['ctrate', 'radgenome']:
    meta = run_meta.get(ds, {}) if isinstance(run_meta.get(ds, {}), dict) else {}
    checks.append((f'{ds}.selected_rows == 450', meta.get('selected_rows') == 450))
    checks.append((f'{ds}.processed_rows == 450', meta.get('processed_rows') == 450))

if validation_rows:
    checks.append(('validation_report all passed', all(row.get('passed', False) for row in validation_rows)))

status_df = pd.DataFrame(checks, columns=['check', 'passed'])
print('
=== 450/450 ????? ===')
display(status_df)
print('PASS count:', int(status_df['passed'].sum()), '/', len(status_df))


## 2. ?????450/450 ????

?? `summary.csv`??????? token ??????????

In [ ]:
summary_csv = OUT_PATH / 'summary.csv'
assert summary_csv.exists(), f'summary.csv not found: {summary_csv}'
summary = pd.read_csv(summary_csv)
print(f'????: {len(summary)}')
if len(summary) != 900:
    print('WARNING: summary.csv row count is not 900; check whether this is really a 450/450 run.')
display(summary.head(3))

agg = summary.groupby('dataset', as_index=False).agg(
    cases=('case_id', 'count'),
    avg_tokens=('n_tokens', 'mean'),
    avg_sentences=('n_sentences', 'mean'),
    avg_violations=('n_violations', 'mean'),
    total_violations=('n_violations', 'sum'),
)
agg['avg_vio_per_sent'] = (agg['avg_violations'] / agg['avg_sentences'].clip(lower=0.01)).round(3)
agg = agg.round({'avg_tokens': 1, 'avg_sentences': 2, 'avg_violations': 3})
print('
=== ????? ===')
display(agg)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics = [
    ('avg_tokens',     'Avg Tokens / Case'),
    ('avg_sentences',  'Avg Sentences / Case'),
    ('avg_violations', 'Avg Violations / Case'),
]
colors = ['#4C72B0', '#DD8452']
for ax, (col, title) in zip(axes, metrics):
    bars = ax.bar(agg['dataset'], agg[col], color=colors[:len(agg)])
    ax.set_title(title)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.02,
                f'{bar.get_height():.2f}',
                ha='center', va='bottom', fontsize=10)
plt.suptitle(f'Run: {OUT_PATH.name}', fontsize=11)
plt.tight_layout()
plt.show()

## 3. Trace 解析（句子级）

遍历所有 `trace.jsonl`，提取 `type=sentence` 行，分析违规规则分布和 `anatomy_keyword`。

In [ ]:
trace_files = sorted((OUT_PATH / 'cases').glob('*/*/trace.jsonl'))
print(f'找到 {len(trace_files)} 个 trace.jsonl')

rows = []
rule_counter: Counter = Counter()

for tf in trace_files:
    dataset = tf.parts[-3]
    case_id = tf.parts[-2]
    with tf.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if obj.get('type') != 'sentence':
                continue
            vios = obj.get('violations') or []
            rule_ids = [
                v['rule_id'] for v in vios
                if isinstance(v, dict) and 'rule_id' in v
            ]
            for rid in rule_ids:
                rule_counter[rid] += 1
            rows.append({
                'dataset': dataset,
                'case_id': case_id,
                'sentence_index': obj.get('sentence_index'),
                'sentence_text': obj.get('sentence_text', ''),
                'anatomy_keyword': obj.get('anatomy_keyword', ''),
                'n_topk': len(obj.get('topk_token_ids') or []),
                'n_violations': len(vios),
                'has_violation': int(len(vios) > 0),
                'rule_ids': '|'.join(rule_ids),
            })

sent_df = pd.DataFrame(rows)
print(f'解析了 {len(sent_df)} 句')
print(f'  有违规的句子: {sent_df["has_violation"].sum()} ({sent_df["has_violation"].mean():.1%})')
display(sent_df.head(5))

In [ ]:
rule_df = pd.DataFrame([
    {'rule_id': k, 'count': v}
    for k, v in sorted(rule_counter.items(), key=lambda x: -x[1])
])
print('=== 规则违规计数 ===')
display(rule_df)

if not rule_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(
        rule_df['rule_id'], rule_df['count'],
        color=sns.color_palette('husl', len(rule_df))
    )
    ax.set_title(f'Violation Count by Rule  ({OUT_PATH.name})')
    ax.set_ylabel('count')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                str(int(bar.get_height())),
                ha='center', va='bottom')
    plt.tight_layout()
    plt.show()

In [ ]:
# R2_ANATOMY 触发最多的 anatomy_keyword
anat_r2 = (
    sent_df[sent_df['rule_ids'].str.contains('R2_ANATOMY', na=False)]
    .groupby('anatomy_keyword')['case_id']
    .count()
    .rename('R2_count')
    .reset_index()
    .sort_values('R2_count', ascending=False)
    .head(20)
)
print('=== R2_ANATOMY top anatomy_keyword ===')
display(anat_r2)

# 各 anatomy_keyword 整体违规率
anat_all = sent_df.groupby('anatomy_keyword', as_index=False).agg(
    n_sentences=('sentence_text', 'count'),
    n_violation_sentences=('has_violation', 'sum'),
)
anat_all['violation_rate'] = (
    anat_all['n_violation_sentences'] / anat_all['n_sentences'].clip(lower=1)
).round(3)
anat_all = anat_all.sort_values('violation_rate', ascending=False)
print('\n=== 各 anatomy_keyword 违规率（top 20）===')
display(anat_all.head(20))

if not anat_r2.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(anat_r2['anatomy_keyword'][::-1], anat_r2['R2_count'][::-1], color='#4C72B0')
    ax.set_title('Top Anatomy Keywords Triggering R2_ANATOMY')
    ax.set_xlabel('violation count')
    plt.tight_layout()
    plt.show()

In [ ]:
sent_agg = sent_df.groupby('dataset', as_index=False).agg(
    sentences=('sentence_text', 'count'),
    violation_sentences=('has_violation', 'sum'),
    total_violations=('n_violations', 'sum'),
)
sent_agg['violation_sentence_rate'] = (
    sent_agg['violation_sentences'] / sent_agg['sentences'].clip(lower=1)
).round(4)
print('=== 句子违规率 (per dataset) ===')
display(sent_agg)

## 4. 病例级分析

按违规率对病例排序，观察分布形态。

In [ ]:
case_df = sent_df.groupby(['dataset', 'case_id'], as_index=False).agg(
    n_sentences=('sentence_text', 'count'),
    n_violation_sentences=('has_violation', 'sum'),
    n_violations_total=('n_violations', 'sum'),
)
case_df['violation_ratio'] = (
    case_df['n_violation_sentences'] / case_df['n_sentences'].clip(lower=1)
).round(4)
case_df = case_df.sort_values(
    ['violation_ratio', 'n_violations_total'], ascending=[False, False]
)
print(f'总病例数: {len(case_df)}')
print(f'violation_ratio=1.0: {(case_df["violation_ratio"] == 1.0).sum()} 例')
print(f'violation_ratio=0.0: {(case_df["violation_ratio"] == 0).sum()} 例')
print('\n=== 违规率最高的 20 个病例 ===')
display(case_df.head(20))

In [ ]:
datasets_present = case_df['dataset'].unique().tolist()
fig, axes = plt.subplots(1, len(datasets_present), figsize=(6 * len(datasets_present), 4))
if len(datasets_present) == 1:
    axes = [axes]
pal = ['#4C72B0', '#DD8452']
for ax, ds, color in zip(axes, datasets_present, pal):
    sub = case_df[case_df['dataset'] == ds]['violation_ratio']
    ax.hist(sub, bins=20, edgecolor='white', color=color, alpha=0.85)
    ax.axvline(sub.mean(), color='red', linestyle='--', label=f'mean={sub.mean():.3f}')
    ax.set_title(f'{ds} — Case Violation Ratio')
    ax.set_xlabel('violation_ratio')
    ax.set_ylabel('# cases')
    ax.legend()
plt.tight_layout()
plt.show()

## 5. Sweep 参数对比（MODE='sweep' 时运行）

对比 `tau_iou × r2_min_support_ratio` 6 组结果，找出最优参数区间。

- `violation_sentence_rate`：越低越好
- `R2_ANATOMY`：R2 触发次数，越低越好
- `R1_LATERALITY` / `R5_NEGATION`：参考指标，应保持稳定（不随 R2 参数变化）

In [ ]:
if MODE != 'sweep':
    print("MODE='single' — 跳过 sweep 对比。改为 MODE='sweep' 后重新运行。")
else:
    def _parse_sweep_run(run_dir: Path) -> dict:
        rm = {}
        rmp = run_dir / 'run_meta.json'
        if rmp.exists():
            rm = json.loads(rmp.read_text(encoding='utf-8'))

        sentence_total = 0
        sentence_with_vio = 0
        rc: Counter = Counter()
        ds_sent: Counter = Counter()
        ds_vio: Counter = Counter()

        cases_dir = run_dir / 'cases'
        if cases_dir.exists():
            for tf in cases_dir.glob('*/*/trace.jsonl'):
                ds = tf.parts[-3]
                with tf.open(encoding='utf-8') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        obj = json.loads(line)
                        if obj.get('type') != 'sentence':
                            continue
                        sentence_total += 1
                        ds_sent[ds] += 1
                        vios = obj.get('violations') or []
                        if vios:
                            sentence_with_vio += 1
                            ds_vio[ds] += 1
                        for v in vios:
                            if isinstance(v, dict):
                                rc[str(v.get('rule_id', 'UNKNOWN'))] += 1

        return {
            'run_dir': run_dir.name,
            'tau_iou': rm.get('tau_iou'),
            'r2_mode': rm.get('r2_mode', 'ratio'),
            'r2_min_support_ratio': rm.get('r2_min_support_ratio'),
            'r4_disabled': rm.get('r4_disabled', False),
            'r5_fallback_disabled': rm.get('r5_fallback_disabled', False),
            'sentence_total': sentence_total,
            'violation_sentence_rate': round(sentence_with_vio / max(1, sentence_total), 4),
            'ctrate_vio_rate': round(ds_vio['ctrate'] / max(1, ds_sent['ctrate']), 4),
            'radgenome_vio_rate': round(ds_vio['radgenome'] / max(1, ds_sent['radgenome']), 4),
            'R1_LATERALITY': rc.get('R1_LATERALITY', 0),
            'R2_ANATOMY': rc.get('R2_ANATOMY', 0),
            'R3_DEPTH': rc.get('R3_DEPTH', 0),
            'R4_SIZE': rc.get('R4_SIZE', 0),
            'R5_NEGATION': rc.get('R5_NEGATION', 0),
            'violations_total': sum(rc.values()),
        }

    _run_dirs = sorted([p for p in SWEEP_PATH.glob(SWEEP_GLOB) if p.is_dir()])
    print(f'Sweep runs ({len(_run_dirs)}) [glob={SWEEP_GLOB}]:')
    for d in _run_dirs:
        print(f'  {d.name}')
    sweep_rows = [_parse_sweep_run(d) for d in _run_dirs]
    sweep_df = pd.DataFrame(sweep_rows).sort_values(
        'violation_sentence_rate', ascending=True
    )
    display(sweep_df)

In [ ]:
if MODE == 'sweep' and 'sweep_df' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # 左：violation_sentence_rate 热图
    piv_rate = sweep_df.pivot_table(
        values='violation_sentence_rate',
        index='tau_iou', columns='r2_min_support_ratio', aggfunc='mean'
    )
    piv_rate.index   = [f'tau={v}' for v in piv_rate.index]
    piv_rate.columns = [f'ratio={v}' for v in piv_rate.columns]
    sns.heatmap(
        piv_rate, annot=True, fmt='.3f', cmap='RdYlGn_r',
        vmin=0.0, vmax=1.0, ax=axes[0], linewidths=0.5
    )
    axes[0].set_title('violation_sentence_rate\n(越低越好)')

    # 右：R2_ANATOMY 计数热图
    piv_r2 = sweep_df.pivot_table(
        values='R2_ANATOMY',
        index='tau_iou', columns='r2_min_support_ratio', aggfunc='sum'
    )
    piv_r2.index   = [f'tau={v}' for v in piv_r2.index]
    piv_r2.columns = [f'ratio={v}' for v in piv_r2.columns]
    sns.heatmap(
        piv_r2, annot=True, fmt='.0f', cmap='YlOrRd',
        ax=axes[1], linewidths=0.5
    )
    axes[1].set_title('R2_ANATOMY count\n(越低越好)')

    plt.suptitle('R2 Sweep: tau_iou × r2_min_support_ratio', fontsize=13, y=1.03)
    plt.tight_layout()
    plt.show()

    best = sweep_df.sort_values('violation_sentence_rate').iloc[0]
    print(f'\n最低 violation_sentence_rate: {best["violation_sentence_rate"]:.4f}')
    print(f'  -> tau_iou={best["tau_iou"]},  r2_min_support_ratio={best["r2_min_support_ratio"]}')

In [ ]:
if MODE == 'sweep' and 'sweep_df' in dir():
    rule_cols = ['R1_LATERALITY', 'R2_ANATOMY', 'R4_SIZE', 'R5_NEGATION']
    labels = [
        f'tau={row.tau_iou}\nratio={row.r2_min_support_ratio}'
        for _, row in sweep_df.iterrows()
    ]
    x = np.arange(len(labels))
    width = 0.2
    fig, ax = plt.subplots(figsize=(14, 5))
    colors = sns.color_palette('husl', len(rule_cols))
    for i, (col, color) in enumerate(zip(rule_cols, colors)):
        ax.bar(x + i * width, sweep_df[col].values, width, label=col, color=color)
    ax.set_xticks(x + width * (len(rule_cols) - 1) / 2)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_title('Rule Violation Count per Sweep Combination')
    ax.set_ylabel('count')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. 病例追踪抽查（Case Inspector）

逐句打印路由结果和违规详情，便于人工核查。

In [ ]:
def inspect_case_trace(dataset_name: str, case_id: str, root: Path = OUT_PATH):
    trace_path = root / 'cases' / dataset_name / case_id / 'trace.jsonl'
    if not trace_path.exists():
        print(f'Not found: {trace_path}')
        return

    with trace_path.open('r', encoding='utf-8') as f:
        data = [json.loads(ln) for ln in f if ln.strip()]

    meta = next((d for d in data if d.get('type') == 'case_meta'), {})
    sentences = [d for d in data if d.get('type') == 'sentence']

    print(f'\n{"="*60}')
    print(f'Case: {case_id}  ({dataset_name})')
    print(
        f'B={meta.get("B")}  k={meta.get("k")}  '
        f'lambda={meta.get("lambda_spatial")}  tau_IoU={meta.get("tau_IoU")}'
    )
    print(f'{"="*60}')

    for s in sentences:
        vios = s.get('violations') or []
        status = 'X' if vios else 'OK'
        anat = s.get('anatomy_keyword', '-')
        text = s.get('sentence_text', '')
        print(f'\n  [{status}] #{s.get("sentence_index")} [{anat}]')
        print(f'       {text}')
        print(f'       cited: {s.get("topk_token_ids")}')
        for v in vios:
            if isinstance(v, dict):
                print(
                    f'       -> [{v.get("rule_id")}] '
                    f'severity={v.get("severity", 0):.3f}  '
                    f'{v.get("message", "")}'
                )


def random_sample_inspect(dataset_name: str, n: int = 3, root: Path = OUT_PATH):
    ds_dir = root / 'cases' / dataset_name
    if not ds_dir.exists():
        print(f'Dataset dir not found: {ds_dir}')
        return
    all_cases = [d.name for d in ds_dir.iterdir() if d.is_dir()]
    if not all_cases:
        print(f'No cases in {ds_dir}')
        return
    sampled = random.sample(all_cases, min(n, len(all_cases)))
    print(f'\n### {dataset_name} — random {len(sampled)} cases ###')
    for case_id in sampled:
        inspect_case_trace(dataset_name, case_id, root)

In [ ]:
# 各随机抽查 2 个；修改 n 增减数量
random_sample_inspect('ctrate',    n=2, root=OUT_PATH)
random_sample_inspect('radgenome', n=2, root=OUT_PATH)

In [ ]:
# 查特定病例（取消注释并填写 case_id）
# inspect_case_trace('ctrate', 'train_10004_a_2', root=OUT_PATH)

## 7. 导出分析结果

将各分析表保存到 `{OUT_PATH}/analysis_exports/`，可直接用于组会或写作。

In [ ]:
export_dir = OUT_PATH / 'analysis_exports'
export_dir.mkdir(parents=True, exist_ok=True)

_saved = []

def _save(df, name):
    p = export_dir / name
    df.to_csv(p, index=False)
    _saved.append(name)

if 'agg'      in dir(): _save(agg,      'dataset_aggregate.csv')
if 'sent_agg' in dir(): _save(sent_agg, 'sentence_violation_rate.csv')
if 'rule_df'  in dir(): _save(rule_df,  'rule_violation_count.csv')
if 'case_df'  in dir(): _save(case_df,  'abnormal_cases_ranked.csv')
if 'case_df'  in dir(): _save(case_df,  'case_violation_ranked.csv')
if 'sent_df'  in dir(): _save(sent_df,  'sentence_detail.csv')
if 'anat_r2'  in dir(): _save(anat_r2,  'anatomy_r2_breakdown.csv')
if 'anat_all' in dir(): _save(anat_all, 'anatomy_all_violation_rate.csv')
if MODE == 'sweep' and 'sweep_df' in dir():
    _save(sweep_df, 'sweep_summary.csv')

print(f'Exported to: {export_dir}')
print('Files:', _saved)
